## Context

Banks incur significant losses due to default in loans. This has led to a tightening up of loan underwriting and has increased loan rejection rates. The need for a better credit risk scoring model is also raised by banks.

The CNK bank has collected customer data for the past few years and wants to build a model to predict if a customer coming to purchase a loan is a good customer (will not default) or a bad customer (will default).


## Data Dictionary

- month - the month of purchase
- credit_amount - amount for which loan is requested
- credit_term - for how long customer wants a loan
- age - age of the customer
- sex - gender of the customer
- education - education level of customer
- product_type - for purchasing what type of product does the customer need a loan (0, 1, 2, 3, 4)
- having_children_flg - if the customer has children or not
- region - customer region category(0, 1, 2)
- income - income of the customer
- family_status - another, married, unmarried
- phone_operator - mobile operator category(0, 1, 2, 3)
- is_client - if the customer wanting to purchase a loan is our client or not
- target - 1-bad customer, 0-good customer




In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV)

from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier

import sklearn.metrics as metrics
from sklearn.metrics import(
    classification_report,
    confusion_matrix,
    recall_score,
    accuracy_score,
    precision_score,
    f1_score)

In [7]:
df = pd.read_csv('Loanclients.csv')
data = df.copy()
data.head(3)

,Month,credit_amount,credit_term,Age,sex,education,product_type,having_children_flg,region,income,family_status,phone_operator,is_client,target
0,10,7000,6,25,0,Secondary special education,1,0,0,21000.0,1,0,0,0
1,10,19000,3,54,0,Secondary special education,3,1,1,17000.0,1,3,0,0
2,1,29000,2,36,0,Secondary special education,1,0,2,31000.0,1,2,0,0


In [8]:
data.info(), data.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Month                1000 non-null   int64  
 1   credit_amount        1000 non-null   int64  
 2   credit_term          1000 non-null   int64  
 3   Age                  1000 non-null   int64  
 4   sex                  1000 non-null   int64  
 5   education            1000 non-null   object 
 6   product_type         1000 non-null   int64  
 7   having_children_flg  1000 non-null   int64  
 8   region               1000 non-null   int64  
 9   income               967 non-null    float64
 10  family_status        1000 non-null   int64  
 11  phone_operator       1000 non-null   int64  
 12  is_client            1000 non-null   int64  
 13  target               1000 non-null   int64  
dtypes: float64(1), int64(12), object(1)
memory usage: 109.5+ KB


(None, (1000, 14))

In [9]:
data.isna().sum()

Month                   0
credit_amount           0
credit_term             0
Age                     0
sex                     0
education               0
product_type            0
having_children_flg     0
region                  0
income                 33
family_status           0
phone_operator          0
is_client               0
target                  0
dtype: int64

* income has some missing data

In [10]:
# these will be broken down for pd.get_dummeies()
data['region'] = data['region'].astype('category')
data['phone_operator'] = data['phone_operator'].astype('category')
data['product_type'] = data['product_type'].astype('category')

In [11]:
data['target'].value_counts()

target
0    893
1    107
Name: count, dtype: int64

* Target field is imbalanced

### Splitting the data into X and y

In [12]:
X = data.drop(['target'], axis=1)
y = data['target']

X = pd.get_dummies(X, drop_first=True, dtype=int)

In [16]:
(X_temp, X_test,
 y_temp, y_test) = train_test_split(X,
                                    y,
                                    test_size=0.2,
                                    random_state=5,
                                    stratify=y)

(X_train, X_val,
 y_train, y_val) = train_test_split(X_temp,
                                    y_temp,
                                    test_size=0.25,
                                    random_state=5,
                                    stratify=y_temp)

print(X_train.shape, X_val.shape, X_test.shape)

(600, 30) (200, 30) (200, 30)


In [17]:
# impute missing values
imp_median = SimpleImputer(missing_values=np.nan, strategy='median')

# fit and transform the imputer on the income test data
X_train['income'] = imp_median.fit_transform(X_train[['income']])

# transform the validation and test income data using
#the imputer fit on training data
X_val['income'] = imp_median.transform(X_val[['income']])
X_test['income'] = imp_median.transform(X_test[['income']])

In [18]:
# Checking class balance for whole data, train set, validation set, and test set

print("Target value ratio in y")
print(y.value_counts(1))
print("*" * 80)
print("Target value ratio in y_train")
print(y_train.value_counts(1))
print("*" * 80)
print("Target value ratio in y_val")
print(y_val.value_counts(1))
print("*" * 80)
print("Target value ratio in y_test")
print(y_test.value_counts(1))
print("*" * 80)

Target value ratio in y
target
0    0.893
1    0.107
Name: proportion, dtype: float64
********************************************************************************
Target value ratio in y_train
target
0    0.893333
1    0.106667
Name: proportion, dtype: float64
********************************************************************************
Target value ratio in y_val
target
0    0.89
1    0.11
Name: proportion, dtype: float64
********************************************************************************
Target value ratio in y_test
target
0    0.895
1    0.105
Name: proportion, dtype: float64
********************************************************************************


## Model evaluation criterion


**What does a bank want?**
* A bank wants to minimize the loss - it can face 2 types of losses here: 
   * Whenever a bank lends money to a customer, they don't return it.
   * A bank doesn't lend money to a customer thinking a customer will default but in reality, the customer won't - opportunity loss.

**Which loss is greater ?**
* Lending to a customer who wouldn't be able to pay back.

**Since we want to reduce loan defaults we should use Recall as a metric of model evaluation instead of accuracy.**

* Recall - It gives the ratio of True positives to Actual positives, so high Recall implies low false negatives, i.e. low chances of predicting a bad customer as a good customer.


# Hyperparameter Tuning

In [19]:
rf = RandomForestClassifier(random_state=1).fit(X_train, y_train)

In [20]:
# Checking recall score on train and validation set
print("Recall on train and validation set")
print(recall_score(y_train, rf.predict(X_train)))
print(recall_score(y_val, rf.predict(X_val)))
print("")

# Checking Precision score on train and validation set
print("Precision on train and validation set")
print(precision_score(y_train, rf.predict(X_train)))
print(precision_score(y_val, rf.predict(X_val)))

print("")

# Checking Accuracy score on train and validation set
print("Accuracy on train and validation set")
print(accuracy_score(y_train, rf.predict(X_train)))
print(accuracy_score(y_val, rf.predict(X_val)))

Recall on train and validation set
1.0
0.5

Precision on train and validation set
1.0
1.0

Accuracy on train and validation set
1.0
0.945


## Grid Search CV

In [21]:
RandomForestClassifier().get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'class_weight': None,
 'criterion': 'gini',
 'max_depth': None,
 'max_features': 'sqrt',
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': None,
 'verbose': 0,
 'warm_start': False}

In [22]:
print(np.arange(0.2, 0.7, 0.1))

print(np.arange(5,10))

[0.2 0.3 0.4 0.5 0.6]
[5 6 7 8 9]


### Tune Random forest using Grid Search

In [24]:
%%time

rf1 = RandomForestClassifier(random_state=1)

parameters = {
    'n_estimators': [150,200,250],
    'min_samples_leaf': np.arange(5,10),
    'max_features': np.arange(0.2,0.7,0.1),
    'max_samples': np.arange(0.3, 0.7, 0.1),
    'class_weight': ['balanced', 'balanced_subsample'],
    'max_depth': np.arange(3,4,5),
    'min_impurity_decrease':[0.001, 0.002, 0.003]}

acc_scorer = metrics.make_scorer(metrics.recall_score)
grid_obj = GridSearchCV(rf1,
                        parameters,
                        scoring=acc_scorer,
                        cv=5,
                        n_jobs=-1,
                        verbose=2).fit(X_train, y_train)
grid_obj.best_params_


Fitting 5 folds for each of 1800 candidates, totalling 9000 fits
CPU times: total: 25.4 s
Wall time: 18min 3s


{'class_weight': 'balanced_subsample',
 'max_depth': 3,
 'max_features': 0.2,
 'max_samples': 0.6000000000000001,
 'min_impurity_decrease': 0.002,
 'min_samples_leaf': 5,
 'n_estimators': 200}

In [25]:
grid_obj.best_score_

0.7192307692307691

In [27]:
rf1_tuned = RandomForestClassifier(
    class_weight='balanced',
    max_features=0.2,
    max_samples=0.6,
    min_samples_leaf=5,
    n_estimators=150,
    max_depth=3,
    random_state=1,
    min_impurity_decrease=0.001).fit(X_train, y_train)

In [28]:
# Checking recall score on train and validation set
print("Recall on train and validation set")
print(recall_score(y_train, rf1_tuned.predict(X_train)))
print(recall_score(y_val, rf1_tuned.predict(X_val)))
print("")

# Checking precision score on train and validation set
print("Precision on train and validation set")
print(precision_score(y_train, rf1_tuned.predict(X_train)))
print(precision_score(y_val, rf1_tuned.predict(X_val)))
print("")

# Checking accuracy score on train and validation set
print("Accuracy on train and validation set")
print(accuracy_score(y_train, rf1_tuned.predict(X_train)))
print(accuracy_score(y_val, rf1_tuned.predict(X_val)))

Recall on train and validation set
0.859375
0.8636363636363636

Precision on train and validation set
0.7971014492753623
0.8260869565217391

Accuracy on train and validation set
0.9616666666666667
0.965


## Randomized Search CV

In [31]:
%%time
rf2 = RandomForestClassifier(random_state=1)
parameters = {
    'n_estimators': [150,200,250],
    'min_samples_leaf': np.arange(5,10),
    'max_features': np.arange(0.2,0.7,0.1),
    'max_samples': np.arange(0.3, 0.7, 0.1),
    'class_weight': ['balanced', 'balanced_subsample'],
    'max_depth': np.arange(3,4,5),
    'min_impurity_decrease':[0.001, 0.002, 0.003]}

acc_scorer = metrics.make_scorer(metrics.recall_score)
grid_obj = RandomizedSearchCV(rf2,
                              parameters,
                              n_iter=30,
                              scoring=acc_scorer,
                              cv=5,
                              random_state=1,
                              n_jobs=-1,
                              verbose=2).fit(X_train, y_train)
grid_obj.best_params_

Fitting 5 folds for each of 30 candidates, totalling 150 fits
CPU times: total: 766 ms
Wall time: 17.5 s


{'n_estimators': 150,
 'min_samples_leaf': 5,
 'min_impurity_decrease': 0.002,
 'max_samples': 0.6000000000000001,
 'max_features': 0.4000000000000001,
 'max_depth': 3,
 'class_weight': 'balanced'}

In [34]:
grid_obj.best_score_

0.6730769230769231

In [35]:
rf2_tuned = RandomForestClassifier(
    class_weight='balanced',
    max_features=0.2,
    max_samples=0.5,
    min_samples_leaf=5,
    n_estimators=150,
    random_state=1,
    max_depth=3,
    min_impurity_decrease=0.003).fit(X_train, y_train)

In [36]:
# Checking recall score on train and validation set
print("Recall on train and validation set")
print(recall_score(y_train, rf2_tuned.predict(X_train)))
print(recall_score(y_val, rf2_tuned.predict(X_val)))
print("")
print("Precision on train and validation set")
# Checking precision score on train and validation set
print(precision_score(y_train, rf2_tuned.predict(X_train)))
print(precision_score(y_val, rf2_tuned.predict(X_val)))
print("")
print("Accuracy on train and validation set")
# Checking accuracy score on train and validation set
print(accuracy_score(y_train, rf2_tuned.predict(X_train)))
print(accuracy_score(y_val, rf2_tuned.predict(X_val)))

Recall on train and validation set
0.859375
0.8636363636363636

Precision on train and validation set
0.7857142857142857
0.8260869565217391

Accuracy on train and validation set
0.96
0.965


#### Choose a best model and predict the performance on the test set

In [37]:
model = rf1_tuned